# Evaluations for finetuned model

In [ ]:
import json
from src.llm_client import OpenAiClient
from src.dataset_gen import generate_judge_dataset
from src.utils import load_dataset_from_jsonl
from src.agent import generate_single_intervention

## Load data

In [ ]:
# Load documents
documents = load_dataset_from_jsonl('results/eval_docs.jsonl')
print(f"Loaded {len(documents)} documents")

In [ ]:
# Load the original evaluation data
interventions_for_eval_baseline = []

with open('results/eval_inference.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        interventions_for_eval_baseline.append(json.loads(line.strip()))

print(f"Loaded {len(interventions_for_eval_baseline)} eval examples")
print(f"\nFirst example keys: {list(interventions_for_eval_baseline[0].keys())}")

# Create a deep copy for GPT-4 interventions (will be modified later)
import copy
interventions_for_eval_gpt4 = copy.deepcopy(interventions_for_eval_baseline)

## Compare finetuned vs baseline

In [ ]:
# Run the judge to compare interventions
judge_llm = OpenAiClient(model="gpt-4.1", temperature=1)

print("Evaluating Baseline vs Finetuned...")
comparison_results_baseline = generate_judge_dataset(
    intervention_pairs=interventions_for_eval_baseline,
    documents=documents,
    judge_llm=judge_llm,
    skip_equal_rating=True,
    agent_output_is_json=True
)
print(f"Generated {len(comparison_results_baseline)} comparison results for baseline vs finetuned")

## Compare finetuned vs GPT-4

In [ ]:
# Replace intervention_1 with GPT-4 generated interventions using the same prompt
# We'll generate new interventions based on the same scenario (doc_index, comments_used, intervention_type)

gpt4_client = OpenAiClient(model="gpt-4.1", temperature=0.8)

from concurrent.futures import ThreadPoolExecutor, as_completed

def generate_intervention_for_pair(pair_with_index):
    i, pair = pair_with_index
    doc_index = pair['doc_index']
    comments_used = pair['comments_used']
    
    # Get the corresponding document
    document = documents[doc_index]
    
    # Generate new intervention_1 using GPT-4
    new_intervention_1 = generate_single_intervention(
        llm=gpt4_client,
        entry=document,
        comments_used=comments_used,
        intervention_type='',
        require_json_output=True
    )
    
    return i, new_intervention_1

# Use ThreadPoolExecutor to parallelize API calls
print("Starting parallel generation of interventions with GPT-4...")
with ThreadPoolExecutor() as executor:
    # Submit all tasks
    future_to_index = {
        executor.submit(generate_intervention_for_pair, (i, pair)): i 
        for i, pair in enumerate(interventions_for_eval_gpt4)
    }
    
    # Process completed tasks and update interventions
    completed = 0
    for future in as_completed(future_to_index):
        i, new_intervention_1 = future.result()
        interventions_for_eval_gpt4[i]['intervention_1'] = new_intervention_1
        
        completed += 1
        if completed % 10 == 0:
            print(f"Processed {completed}/{len(interventions_for_eval_gpt4)} interventions")

print(f"\nCompleted generating {len(interventions_for_eval_gpt4)} new interventions with GPT-4")

In [ ]:
print("\nEvaluating GPT-4 vs Finetuned...")
comparison_results_gpt4 = generate_judge_dataset(
    intervention_pairs=interventions_for_eval_gpt4,
    documents=documents,
    judge_llm=judge_llm,
    skip_equal_rating=True,
    agent_output_is_json=True
)
print(f"Generated {len(comparison_results_gpt4)} comparison results for GPT-4 vs finetuned")

## Analysis

In [ ]:
# Define function to analyze and report win rates
import pandas as pd
import matplotlib.pyplot as plt

def analyze_win_rates(comparison_results, interventions_for_eval, model1_name, model2_name, save_prefix):
    """
    Analyze win rates between two models.
    
    Args:
        comparison_results: List of comparison results from judge
        interventions_for_eval: List of intervention pairs
        model1_name: Name of model 1 (intervention_1)
        model2_name: Name of model 2 (intervention_2)
        save_prefix: Prefix for saving visualization files
    """
    df_comparison = pd.DataFrame(comparison_results)
    df_interventions = pd.DataFrame(interventions_for_eval)
    
    # Merge the intervention types into the comparison dataframe
    df_comparison = df_comparison.merge(
        df_interventions[['doc_index', 'comments_used', 'intervention_1_type', 'intervention_2_type']], 
        on=['doc_index', 'comments_used'],
        how='left'
    )
    
    print("\n" + "="*60)
    print(f"EVALUATION RESULTS: {model1_name} vs {model2_name}")
    print("="*60)
    
    MAPPING = {
        1: model1_name,
        2: model2_name
    }
    
    # Overall win rate
    winner_counts = df_comparison['accepted_agent'].value_counts()
    total = len(df_comparison)
    
    print(f"\nTotal comparisons: {total}")
    print(f"\nOverall win statistics:")
    for winner, count in winner_counts.items():
        percentage = (count / total) * 100
        print(f"  {MAPPING[winner]}: {count} ({percentage:.1f}%)")
    
    model1_win_rate_overall = (winner_counts.get(1, 0) / total) * 100
    model2_win_rate_overall = (winner_counts.get(2, 0) / total) * 100
    
    print(f"\n" + "="*60)
    print(f"OVERALL WIN RATES")
    print(f"{model2_name.upper()}: {model2_win_rate_overall:.1f}%")
    print(f"{model1_name.upper()}: {model1_win_rate_overall:.1f}%")
    print("="*60)
    
    # Overall win rate excluding cases where either model selected no_intervention
    print("\n" + "="*60)
    print("OVERALL WIN RATE (EXCLUDING NO_INTERVENTION CASES)")
    print("="*60)
    
    # Add intervention columns to comparison dataframe if not already there
    if 'intervention_1' not in df_comparison.columns:
        df_comparison = df_comparison.merge(
            df_interventions[['doc_index', 'comments_used', 'intervention_1', 'intervention_2']], 
            on=['doc_index', 'comments_used'],
            how='left'
        )
    
    df_no_no_intervention = df_comparison[
        (~df_comparison['intervention_1'].str.contains('NO_INTERVENTION', na=False)) & 
        (~df_comparison['intervention_2'].str.contains('NO_INTERVENTION', na=False))
    ]

    
    winner_counts_filtered = df_no_no_intervention['accepted_agent'].value_counts()
    total_filtered = len(df_no_no_intervention)
    
    print(f"\nTotal comparisons (excluding no_intervention): {total_filtered}")
    print(f"Excluded comparisons: {len(df_comparison) - total_filtered}")
    
    if total_filtered > 0:
        print(f"\nWin statistics:")
        for winner, count in winner_counts_filtered.items():
            percentage = (count / total_filtered) * 100
            print(f"  {MAPPING[winner]}: {count} ({percentage:.1f}%)")
        
        model1_win_rate = (winner_counts_filtered.get(1, 0) / total_filtered) * 100
        model2_win_rate = (winner_counts_filtered.get(2, 0) / total_filtered) * 100
        
        print(f"\n" + "="*60)
        print(f"{model2_name.upper()} WIN RATE: {model2_win_rate:.1f}%")
        print(f"{model1_name.upper()} WIN RATE: {model1_win_rate:.1f}%")
        print("="*60)
    else:
        print("\nNo comparisons available after excluding no_intervention cases")
    
    # Visualize results
    fig, ax = plt.subplots(figsize=(4, 4))
    
    wedges, texts, autotexts = ax.pie(winner_counts.values, labels=[MAPPING[w] for w in winner_counts.index],
                                        autopct='%1.1f%%', startangle=90)
    ax.set_title(f'Win Rate Distribution: {model1_name} vs {model2_name}')
    plt.tight_layout()
    plt.savefig(f'results/{save_prefix}_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nVisualization saved to results/{save_prefix}_comparison.png")
    
    # Show some examples where model2 wins
    print("\n" + "="*80)
    print(f"EXAMPLES WHERE {model2_name.upper()} WINS")
    print("="*80)
    
    model2_wins = df_comparison[df_comparison['accepted_agent'] == 2].head(5)
    for idx, row in model2_wins.iterrows():
        print(f"\n--- Example {idx + 1} ---")
        print(f"Document index: {row['doc_index']}, Comments used: {row['comments_used']}")
        print(f"\n{model2_name} (Accepted):\n{row['accepted']}")
        print(f"\n{model1_name} (Rejected):\n{row['rejected']}")
        print("\n" + "-"*80)
    
    # Show some examples where model1 wins
    print("\n" + "="*80)
    print(f"EXAMPLES WHERE {model1_name.upper()} WINS")
    print("="*80)
    
    model1_wins = df_comparison[df_comparison['accepted_agent'] == 1].head(5)
    for idx, row in model1_wins.iterrows():
        print(f"\n--- Example {idx + 1} ---")
        print(f"Document index: {row['doc_index']}, Comments used: {row['comments_used']}")
        print(f"\n{model1_name} (Accepted):\n{row['accepted']}")
        print(f"\n{model2_name} (Rejected):\n{row['rejected']}")
        print("\n" + "-"*80)
    
    return df_comparison

print("Win rate analysis function defined")

In [ ]:
# Analyze Baseline vs Finetuned
df_baseline = analyze_win_rates(
    comparison_results_baseline,
    interventions_for_eval_baseline,
    model1_name="Baseline",
    model2_name="Finetuned",
    save_prefix="baseline_vs_finetuned"
)

In [ ]:
# Analyze GPT-4 vs Finetuned
df_gpt4 = analyze_win_rates(
    comparison_results_gpt4,
    interventions_for_eval_gpt4,
    model1_name="GPT-4",
    model2_name="Finetuned",
    save_prefix="gpt4_vs_finetuned"
)

In [ ]:
# Win Rate Analysis: Baseline vs GPT-4.1
print("="*80)
print("BASELINE VS GPT-4.1 WIN RATE ANALYSIS")
print("="*80)

# Create pairs comparing baseline (intervention_1) vs gpt4 (intervention_1)
# We'll use the judge to compare these two directly
baseline_vs_gpt4_pairs = []

for i in range(len(interventions_for_eval_baseline)):
    baseline_pair = interventions_for_eval_baseline[i]
    gpt4_pair = interventions_for_eval_gpt4[i]
    
    # Create a new comparison pair with baseline as intervention_1 and gpt4 as intervention_2
    new_pair = {
        'doc_index': baseline_pair['doc_index'],
        'scenario': baseline_pair['scenario'],
        'comments_used': baseline_pair['comments_used'],
        'intervention_1': baseline_pair['intervention_1'],
        'intervention_1_type': baseline_pair['intervention_1_type'],
        'intervention_2': gpt4_pair['intervention_1'],  # GPT-4's intervention
        'intervention_2_type': gpt4_pair['intervention_1_type']
    }
    baseline_vs_gpt4_pairs.append(new_pair)

print(f"\nCreated {len(baseline_vs_gpt4_pairs)} comparison pairs")

# Run the judge to compare baseline vs gpt4
print("\nEvaluating Baseline vs GPT-4.1...")
comparison_results_baseline_vs_gpt4 = generate_judge_dataset(
    intervention_pairs=baseline_vs_gpt4_pairs,
    documents=documents,
    judge_llm=judge_llm,
    skip_equal_rating=True,
    agent_output_is_json=True
)

print(f"\nGenerated {len(comparison_results_baseline_vs_gpt4)} comparison results")

# Save results
with open('results/evaluation_baseline_vs_gpt4.jsonl', 'w', encoding='utf-8') as f:
    for item in comparison_results_baseline_vs_gpt4:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print("Results saved to results/evaluation_baseline_vs_gpt4.jsonl")

# Analyze the results
df_baseline_vs_gpt4 = analyze_win_rates(
    comparison_results_baseline_vs_gpt4,
    baseline_vs_gpt4_pairs,
    model1_name="Baseline",
    model2_name="GPT-4.1",
    save_prefix="baseline_vs_gpt4"
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Extract intervention types from each model
def extract_intervention_types(intervention_list, intervention_key='intervention_1'):
    """Extract intervention types from a list of interventions"""
    types = []
    for item in intervention_list:
        try:
            intervention = item.get(intervention_key, '')
            if isinstance(intervention, str):
                intervention_data = json.loads(intervention)
            else:
                intervention_data = intervention

            intervention_type = intervention_data.get('intervention_type', 'unknown')
            types.append(intervention_type)
        except:
            types.append('unknown')
    return types

# Get intervention types for each model
baseline_types = extract_intervention_types(interventions_for_eval_baseline, 'intervention_1')
finetuned_types = extract_intervention_types(interventions_for_eval_baseline, 'intervention_2')
gpt4_types = extract_intervention_types(interventions_for_eval_gpt4, 'intervention_1')

# Count intervention types
baseline_counts = Counter(baseline_types)
finetuned_counts = Counter(finetuned_types)
gpt4_counts = Counter(gpt4_types)

# Get all unique intervention types
all_types = sorted(set(baseline_types + finetuned_types + gpt4_types))

# Map intervention types to cleaner labels
type_labels = {
    'no_intervention': 'No Intervention',
    'socratic': 'Socratic Question',
    'compromise': 'Compromise Synthesis',
    'unknown': 'Unknown'
}

clean_labels = [type_labels.get(t, t.replace('_', ' ').title()) for t in all_types]

# Prepare data for grouped bar chart
baseline_data = [baseline_counts.get(t, 0) for t in all_types]
finetuned_data = [finetuned_counts.get(t, 0) for t in all_types]
gpt4_data = [gpt4_counts.get(t, 0) for t in all_types]

# Create the histogram
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(all_types))
width = 0.25

bars1 = ax.bar(x - width, baseline_data, width, label='Baseline', alpha=0.8, color='#3498db')
bars2 = ax.bar(x, finetuned_data, width, label='Finetuned', alpha=0.8, color='#2ecc71')
bars3 = ax.bar(x + width, gpt4_data, width, label='GPT-4', alpha=0.8, color='#e74c3c')

# Customize the plot
ax.set_xlabel('Intervention Type', fontsize=12, fontweight='bold')
ax.set_ylabel('Count', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Intervention Types by Model', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(clean_labels, rotation=0, ha='center')
ax.legend(loc='upper right', fontsize=10)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
def add_value_labels(bars):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height)}',
                   ha='center', va='bottom', fontsize=9)

add_value_labels(bars1)
add_value_labels(bars2)
add_value_labels(bars3)

plt.tight_layout()
plt.savefig('results/intervention_types_histogram.png', dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("="*60)
print("INTERVENTION TYPE DISTRIBUTION")
print("="*60)
print(f"\n{'Intervention Type':<25} {'Baseline':<12} {'Finetuned':<12} {'GPT-4':<12}")
print("-"*60)
for i, t in enumerate(all_types):
    label = type_labels.get(t, t.replace('_', ' ').title())
    print(f"{label:<25} {baseline_data[i]:<12} {finetuned_data[i]:<12} {gpt4_data[i]:<12}")
print("-"*60)
print(f"{'Total':<25} {sum(baseline_data):<12} {sum(finetuned_data):<12} {sum(gpt4_data):<12}")
print("="*60)

# Calculate percentages
print("\n" + "="*60)
print("INTERVENTION TYPE PERCENTAGES")
print("="*60)
print(f"\n{'Intervention Type':<25} {'Baseline':<12} {'Finetuned':<12} {'GPT-4':<12}")
print("-"*60)
total_baseline = sum(baseline_data)
total_finetuned = sum(finetuned_data)
total_gpt4 = sum(gpt4_data)

for i, t in enumerate(all_types):
    label = type_labels.get(t, t.replace('_', ' ').title())
    baseline_pct = (baseline_data[i] / total_baseline * 100) if total_baseline > 0 else 0
    finetuned_pct = (finetuned_data[i] / total_finetuned * 100) if total_finetuned > 0 else 0
    gpt4_pct = (gpt4_data[i] / total_gpt4 * 100) if total_gpt4 > 0 else 0
    print(f"{label:<25} {baseline_pct:<11.1f}% {finetuned_pct:<11.1f}% {gpt4_pct:<11.1f}%")
print("="*60)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Extract intervention types from each model
def extract_intervention_types(intervention_list, intervention_key='intervention_1'):
    """Extract intervention types from a list of interventions"""
    types = []
    for item in intervention_list:
        try:
            intervention = item.get(intervention_key, '')
            if isinstance(intervention, str):
                intervention_data = json.loads(intervention)
            else:
                intervention_data = intervention

            intervention_type = intervention_data.get('intervention_type', 'unknown')
            types.append(intervention_type)
        except:
            types.append('unknown')
    return types

# Get intervention types for each model
baseline_types = extract_intervention_types(interventions_for_eval_baseline, 'intervention_1')
finetuned_types = extract_intervention_types(interventions_for_eval_baseline, 'intervention_2')
gpt4_types = extract_intervention_types(interventions_for_eval_gpt4, 'intervention_1')

# Count intervention types
baseline_counts = Counter(baseline_types)
finetuned_counts = Counter(finetuned_types)
gpt4_counts = Counter(gpt4_types)

# Get all unique intervention types
all_types = sorted(set(baseline_types + finetuned_types + gpt4_types))

# Map intervention types to cleaner labels
type_labels = {
    'no_intervention': 'No Intervention',
    'socratic': 'Socratic Question',
    'compromise': 'Compromise Synthesis',
    'unknown': 'Invalid'
}

clean_labels = [type_labels.get(t, t.replace('_', ' ').title()) for t in all_types]

# Prepare data for grouped bar chart
baseline_data = [baseline_counts.get(t, 0) for t in all_types]
finetuned_data = [finetuned_counts.get(t, 0) for t in all_types]
gpt4_data = [gpt4_counts.get(t, 0) for t in all_types]

# Create the histogram
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(all_types))
width = 0.25

bars1 = ax.bar(x - width, baseline_data, width, label='Baseline', alpha=0.8, color='#3498db')
bars2 = ax.bar(x, finetuned_data, width, label='Finetuned', alpha=0.8, color='#2ecc71')
# bars3 = ax.bar(x + width, gpt4_data, width, label='GPT-4', alpha=0.8, color='#e74c3c')

# Customize the plot
ax.set_xlabel('Intervention Type', fontsize=12, fontweight='bold')
ax.set_ylabel('Count', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Intervention Types by Model in Eval Dataset', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(clean_labels, rotation=0, ha='center')
ax.legend(loc='upper right', fontsize=10)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
def add_value_labels(bars):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height)}',
                   ha='center', va='bottom', fontsize=9)

add_value_labels(bars1)
add_value_labels(bars2)
# add_value_labels(bars3)

plt.tight_layout()
plt.savefig('results/intervention_types_histogram.png', dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("="*60)
print("INTERVENTION TYPE DISTRIBUTION")
print("="*60)
print(f"\n{'Intervention Type':<25} {'Baseline':<12} {'Finetuned':<12} {'GPT-4':<12}")
print("-"*60)
for i, t in enumerate(all_types):
    label = type_labels.get(t, t.replace('_', ' ').title())
    print(f"{label:<25} {baseline_data[i]:<12} {finetuned_data[i]:<12} {gpt4_data[i]:<12}")
print("-"*60)
print(f"{'Total':<25} {sum(baseline_data):<12} {sum(finetuned_data):<12} {sum(gpt4_data):<12}")
print("="*60)

# Calculate percentages
print("\n" + "="*60)
print("INTERVENTION TYPE PERCENTAGES")
print("="*60)
print(f"\n{'Intervention Type':<25} {'Baseline':<12} {'Finetuned':<12} {'GPT-4':<12}")
print("-"*60)
total_baseline = sum(baseline_data)
total_finetuned = sum(finetuned_data)
total_gpt4 = sum(gpt4_data)

for i, t in enumerate(all_types):
    label = type_labels.get(t, t.replace('_', ' ').title())
    baseline_pct = (baseline_data[i] / total_baseline * 100) if total_baseline > 0 else 0
    finetuned_pct = (finetuned_data[i] / total_finetuned * 100) if total_finetuned > 0 else 0
    gpt4_pct = (gpt4_data[i] / total_gpt4 * 100) if total_gpt4 > 0 else 0
    print(f"{label:<25} {baseline_pct:<11.1f}% {finetuned_pct:<11.1f}% {gpt4_pct:<11.1f}%")
print("="*60)